# Task 11 — Exception Handling

This task sheet is the hands-on companion to Session 11. It contains six problems, and every single one is built around the same idea: **a program should fail gracefully instead of crashing**. In normal Python, when something goes wrong — an index out of range, a value that cannot be converted, a key that does not exist, a division by zero — the interpreter raises an **exception** and stops the program unless you catch it. The whole session taught you the machinery for controlling that moment: `try` and `except` to intercept the error, `else` to run code only when the `try` block succeeded, `finally` to run cleanup no matter what, and `raise` to create brand-new errors of your own design.

Before you start, keep a mental checklist of the components you will use. Every `try` block may be followed by as many `except` clauses as you like, each naming one exception type and the code to handle just that type. Order matters: Python checks the except clauses top to bottom and catches the first one whose type matches, so always list more specific exceptions before more general ones. The `else` clause runs only if no exception occurred — it is the natural place for the "success message" a problem asks you to keep out of the exception handler. The `finally` clause runs unconditionally, even when a `return` or a `break` is executed, which makes it ideal for printing a closing message or closing a resource. And remember that even `return` inside `try` does not skip `finally`.

Two further points will recur. First, Python's built-in exception classes form a hierarchy: `Exception` is the parent of most of them, so you can write one handler for `Exception` to catch almost anything, or several handlers — `IndexError`, `ValueError`, `TypeError`, `KeyError`, `ZeroDivisionError` — to treat each case separately. Second, `raise` lets you throw your own exceptions, including custom classes you define by subclassing `Exception`, which is exactly what the number-game and voting problems require. Work through the six questions in order; each one reuses the vocabulary from the previous. Good luck, and let the errors teach you.

###`Q-1`: You are given a function definition. There might be several issues on execution of this function. You are asked to do exception handling for diffrent errors that this function goes in to `without altering this function`. And print error text.



Function parameters `l -> list, s -> could be anything`

```
def function(l: list, s, **args):
    last_element = l[-1]
    
    l[int(s)]=10
    any_element = l[int(s)+10]
    l[s]=10
    
    res = sum(l)
    
    p = args['p']
    # print(p)
    return res/last_element * p + any_element

```
Check for different function calls:-

```
function([1,2,1], 12)
function([1,2,1]*9, '1-2')
function([1,'2',1]*9, 12)
function([1,'2',1]*9, 12)
function([1,2,0]*9, 12  )
function([1,2,1]*9, 12, p=None)
function([1,2,0]*9, 12, p=10)
```

## Q1 — wrapping a fixed, unknown function in error handlers

This problem gives you a function `function(l, s, **args)` that you are not allowed to change. Your job is to call it the way a user might and make sure no call ever crashes the notebook — instead, each failure should be caught and its **error text printed**. To prepare, recall which built-in exceptions a function like this can produce:

- **IndexError** — raised when `l[int(s)]` or `l[int(s)+10]` uses an index outside the list, and also when `l[-1]` fails on an empty list.
- **ValueError** — raised when `int(s)` is called on a string that cannot be parsed, for example `int('1-2')`.
- **TypeError** — raised when you index a list with a non-integer (try `l[s]` with a string) or when `sum(l)` meets its own error conditions.
- **KeyError** — raised by `args['p']` when no keyword argument named `p` was passed.
- **ZeroDivisionError** — raised by `res / last_element` when `last_element` is `0`.

Your solution should therefore define the function exactly as given, then make a series of calls — each one wrapped so that any of the above can be intercepted. Print the type of the exception and its message so you can see what the interpreter actually said.

### Hints
- Use a `try:` / `except ... as e:` block; the syntax `print(type(i))` and `print(i)` shows both the class and the message.
- A list of the seven suggested calls is provided in the question — run them all, one per guarded block, or loop over parameter tuples.
- Remember the order of `except` clauses: put specific exceptions like `KeyError` and `ZeroDivisionError` before the generic ones.
- To keep the output readable, print a small caption such as the parameter values before each call's result.

In [ ]:
# Write code here

###`Q-2:` You are given a code snippet. There might be several issues on execution of this code. You are asked to do exception handling for diffrent errors, condition is what ever happens we need to execute last line printing correct result of `sum of elements`.

List have elemnts as any no of  `key-pair dict with key as list index and value as any integer`, `integers` and `numeric-strings`. There is always only one element in the dict.


```
l = [{0:2},2,3,4,'5', {5:10}]
# For calculating sum of above list
s=0
for i in range(len(l)):
    #You can Edit code from here
    s += l[i].get(i)
    s += l[i]
    s += int(l[i])


print(s)
```

## Q2 — guaranteed sum over a mixed `list`, no matter what is inside

The second question hands you a list `l` whose elements are a mixture of **plain integers**, **numeric strings** like `'5'`, and **dicts with exactly one key–value pair** like `{0: 2}`. You must compute the correct total and still reach the final `print(s)` even though naive code would crash on the very first incompatible element. This is a classic exercise in **nested exception handling**: one error is handled, and inside that handler another, separate error may still occur.

Think about what each operation raises:

- `l[i]` on a plain integer `2` works fine; on the string `'5'` it also works (returning the string). On the dict `{0: 2}` it **raises TypeError**, because the dict is indexed with an integer key `i`, and `i` happens to be `0` for the first dict but `5` for the last — so the value exists for the first dict and is missing for the last.
- For a dict element, `l[i].get(i)` extracts the value by key, but calling `.get` on an int or a string raises **AttributeError**.
- `int(l[i])` converts numeric strings to integers, completing the fallback chain.

The elegant structure the expected solution uses is a chain of `try/except` blocks nested inside one another: first try the plain addition; if that fails, try the dictionary `.get` route; if that fails, convert to `int`. When a level succeeds you simply break out of the loop so the fallbacks do not run — or rely on `try` falling through only when an exception was raised.

### Hint
- Verify by hand that the correct total for `l = [{0:2}, 2, 3, 4, '5', {5:10}]` is `26` (2 + 2 + 3 + 4 + 5 + 10).

In [ ]:
# Write code here

### `Q-3:`: File Handling with Exception handling

Write a program that opens a text file and write data to it as "Hello, Good Morning!!!". Handle exceptions that can be generated during the I/O operations. Do not show the success message on the main exception handling block (write inside the else block).

## Q3 — file I/O wrapped in try, with the success message in `else`

Here the topic changes from arithmetic to **exceptions during input/output**. Writing a file can fail for reasons outside your program — a full disk, a missing directory, no write permission — and Python wraps all of these in **IOError** (an alias for the newer **OSError**). This question asks you to open a text file, write the phrase `Hello, Good Morning!!!`, and catch whatever I/O problem occurs, while following one very specific rule: **the success message must live in the `else` clause, not in the exception handler and not after the `with` block**.

Why does this matter? The `try/except/else/finally` shape gives you three clearly separated zones. The `try` block contains the risky operation. The `except` block handles the failure case — the "something went wrong" message. The `else` block is the positive counterpart: code that should run **only when the try block completed without raising**. Putting the success message inside `else` guarantees that a real I/O failure never prints a misleading success line, because on failure the flow jumps directly to `except` and skips `else` entirely.

Because you are using a `with open(...) as f:` context manager, the file is closed automatically even if writing raises — no explicit `f.close()` is needed, and there is no `finally` required here. The expected output is simply `File written successfully`.

### Hints
- Catch the broad exception with `except IOError:` (or `except OSError:`); keep the message short.
- Put `print("File written successfully")` inside the `else:` — this is the point of the exercise.
- Run the code twice: once normally, and once after making the path invalid (for example a non-existent folder) to confirm the exception branch fires instead of `else`.

In [ ]:
# write code here

### `Q-4`: Number game program.

Write a number game program. Ask the user to enter a number. If the number is greater than number to be guessed, raise a **ValueTooLarge** exception. If the value is smaller the number to be guessed the, raise a **ValueTooSmall** exception and prompt the user to enter again. Quit the program only when the user enters the correct number. Also raise **GuessError** if user guess a number less than 1.

## Q4 — custom exceptions and a repeating game loop

This problem introduces your first **custom exception classes** and an indefinite loop that only ends when the user guesses perfectly. The game picks a secret number (the sample below uses `random.randint(1, 100)`), then asks the user to enter guesses. If the guess is too small you must raise `ValueTooSmall`; if too large, `ValueTooLarge`; if the user enters a number below 1, raise `GuessError`. The program keeps prompting until the user nails it, then exits.

To build a custom exception you subclass the standard base:

```python
class ValueTooLarge(Exception):
    def display(self):
        print("Input value is too large")
```

The class inherits everything `Exception` offers; the extra `display()` method is just a convenient way to print a friendly message from the handler (`except ValueTooLarge as e: e.display()`). Defining separate classes per situation lets you write **distinct, targeted handlers** instead of one `except Exception:` and then testing the message text — this is the whole reason Session 11 recommended custom exceptions over generic ones.

The loop structure to remember: everything that can fail — converting the input with `int(input(...))` and performing the three comparisons — sits inside a single `try`. The `raise` statements send the flow to the matching `except` clauses, which print the hint, and the loop continues. Only when `guess == num` does the code run the success `print` and `break`, ending the loop.

### Hints
- Put `num = random.randint(1, 100)` once, **before** the loop.
- Wrap the input and comparisons in `try:`, then handle `ValueTooSmall`, `ValueTooLarge`, and `GuessError` with three separate handlers.
- `ValueToo...` here means *relative to the target number*: a below-target guess raises `ValueTooSmall`.
- The `input` conversion can also raise bare `ValueError` if the user types letters — catching custom exceptions only is acceptable for this exercise, but mention the risk.

In [ ]:
# Write code here

### `Q-5:` Cast vote

Write a program that validate name and age as entered by the user to determine whether the person can cast vote or not. To handle the age, create **InvalidAge** exception and for name, create **InvalidName** exception. The name will be invalid when the string will be empty or name has only one word.

Example 1:

Input:

```bash
Enter the name:               goransh singh
Enter the age: 25
```

Output:

```bash
Goransh Singh  Congratulation !!! You can vote.
```

## Q5 — validating two inputs: a name and an age

Now you combine custom exceptions with **input validation** for two independent fields. The task: read a name and an age, decide whether the person may cast a vote, and raise `InvalidName` when the name is empty or contains only one word, or `InvalidAge` when the age is below 18. When both checks pass, print the congratulations message; the problem is designed so you see different *branches* of the same `try/except` skeleton.

Think about what decides each error:

- A **valid name** must have at least two words, so `len(name.split()) < 2` is the failing test — note how `.split()` conveniently separates words regardless of extra spaces. An empty string also fails this test, so you do not even need a separate empty check, although you may keep both for clarity.
- A **valid age** must be at least 18, so `int(input(...))` should also be protected, because a non-numeric answer raises a built-in `ValueError` you did not design. The corrected sample input `"goransh singh"` and `25` prints the acceptance message.

This is also your chance to use the full four-part structure in one program: `try` holds the reading and checks, `except InvalidName` and `except InvalidAge` hold the two rejection messages, `else` prints "your vote is accepted", and `finally` prints a closing thank-you line — demonstrating that `finally` runs whether the vote was accepted or rejected.

### Hints
- Validate the name via word count (`split()`), not character count.
- Convert the age with `int(...)` inside the `try` so a bad number is caught too.
- Remember the `else` clause fires only when the entire `try` ran cleanly.

In [ ]:
# Write code here

### `Q-6`: Write a python function which infinitely prints natural numbers in a single line. Raise the **StopIteration** exception after displaying first 20 numnbers to exit from the program.

## Q6 — StopIteration to end an infinite generator-style loop

The final question looks unusual: write a function that would print natural numbers forever, but make it stop after exactly twenty outputs by **raising `StopIteration` manually**. Normally `StopIteration` is raised internally by iterators to signal "no more items" — a `for` loop treats it as the normal end of iteration. Here you are asked to *borrow* that signal inside your own loop and use it to break out.

A clean way to think about the code: keep a counter. At each iteration, increment it; when it reaches the threshold, `raise StopIteration`. Because the function already contains a `try`, the matching `except StopIteration:` catches the exception you just raised and `break`s out of the otherwise-infinite `while True` loop. The `else` clause then prints the current number normally — so numbers are only printed on the iterations that do *not* end the program. This mirrors the constellation: `try` (increment + possibly raise), `except` (catching the signal to exit), `else` (the print that must not run on the final iteration).

Starting from 100, the expected output is `101 102 ... 120` across one line — exactly twenty numbers. The trick is that the first number printed comes from `n += 1` with `n` initially 100, and the counter tested before printing guarantees the exit happens after, not before, the twentieth number.

### Hints
- Decide the exact check: raising when the iteration counter equals 21 means twenty successful prints.
- Keep the `print(n, end=' ')` inside `else` so the terminal number is never printed after the break.
- Raise, catch, and break inside the same guard block is fine — the point is to demonstrate a *manual* `StopIteration`.

## Summary

This sheet applied every exception-handling tool from Session 11 to six realistic programs. In Q1 you wrapped a fixed, error-prone function and learned that each built-in error — `IndexError`, `ValueError`, `TypeError`, `KeyError`, `ZeroDivisionError` — has a distinct trigger you can predict. In Q2 you nested `try/except` chains to sum a heterogeneous list without ever crashing. Q3 showed the value of `else` when handling files: the success message belongs there, and only there. Q4 and Q5 introduced **custom exception classes** that subclass `Exception`, letting you model domain rules like `ValueTooLarge`, `GuessError`, `InvalidName`, or `InvalidAge` and write precise, readable handlers. Finally, Q6 demonstrated that `StopIteration` is just one more exception: raising it manually and catching it with a matching `except` block gives you a clean way to end an otherwise-infinite loop. Across all six you saw the same skeleton — `try`, `except`, `else`, `finally`, `raise` — bent to different problems, which is exactly how professional Python code keeps programs stable when the unexpected happens.

In [ ]:
# Write code here